# 02 - Feature Engineering

Goal: build the early-window feature table (day-1/day-2 cutoff only) for
each of the 607 events, join it against the `severity_class` labels from
`01_data_ingestion.ipynb`, and produce a single train-ready DataFrame for
the GBM (XGBoost) and EBM (InterpretML) baselines in Week 2.

Everything here reads ONLY from the early cutoff window -- reaching into
later days would leak the label, since severity_class is derived from the
full trajectory.

In [1]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from flashpoint.db import get_connection

DB_PATH = Path("../data/flashpoint.duckdb")
CUTOFF_DAY = 2  # use the first 2 days as the "early" window

con = get_connection(DB_PATH)
events_df = con.execute("SELECT * FROM events").df()
outcomes_df = con.execute("SELECT * FROM event_outcomes").df()
print(f"{len(events_df)} events, {len(outcomes_df)} outcomes")

607 events, 607 outcomes


## Step 1: Filter out events too short for the cutoff

An event needs at least `CUTOFF_DAY` days of observations to compute early
features at all -- check for any that don't, per the note flagged at the
end of `01_data_ingestion.ipynb`.

In [2]:
too_short = events_df[events_df['n_days'] < CUTOFF_DAY]
print(f"{len(too_short)} events have fewer than {CUTOFF_DAY} days -- these get dropped")

usable_events = events_df[events_df['n_days'] >= CUTOFF_DAY].copy()
print(f"{len(usable_events)} usable events remain")

0 events have fewer than 2 days -- these get dropped
607 usable events remain


## Step 2: Compute early-window features per event

In [3]:
import pandas as pd
from tqdm import tqdm
from flashpoint.data_access import HDF5Event, read_event_window
from flashpoint.features import early_window_stats

feature_rows = []
for row in tqdm(usable_events.itertuples(), total=len(usable_events)):
    event = HDF5Event(
        event_id=row.event_id, year=row.year, hdf5_path=Path(row.hdf5_path),
        n_days=row.n_days, img_dates=[], lnglat=(row.centroid_lon, row.centroid_lat),
    )
    early_stack = read_event_window(event, 0, CUTOFF_DAY)
    stats = early_window_stats(early_stack)
    stats["event_id"] = row.event_id
    feature_rows.append(stats)

features_df = pd.DataFrame(feature_rows)
features_df.head()

100%|██████████| 607/607 [00:05<00:00, 108.51it/s]


,fire_extent_ha,wind_speed_mean,wind_speed_max,wind_direction_sin_mean,wind_direction_cos_mean,max_temp_max,min_temp_min,humidity_min,pdsi_mean,erc_mean,event_id
0,0.0,0.895985,2.4,-0.354610,-0.385744,294.899994,274.100006,0.00178,-1.642182,52.257618,fire_21458798
1,0.0,4.423371,9.5,-0.023865,-0.802851,292.399994,259.299988,0.00131,-2.819414,61.060646,fire_21458801
2,0.0,1.746247,4.0,0.027330,0.047859,295.200012,272.100006,0.00349,-1.862331,26.161974,fire_21458806
3,112.5,1.051939,2.6,-0.341254,-0.491015,299.399994,271.799988,0.00073,-1.110331,64.648163,fire_21458836
4,0.0,4.598395,9.1,-0.122574,-0.069806,287.000000,250.300003,0.00114,-2.819055,49.521889,fire_21458848


## Step 3: Write early_features to DuckDB, then join with labels

In [4]:
insert_rows = [
    (
        r["event_id"], CUTOFF_DAY, r["fire_extent_ha"], r["wind_speed_mean"],
        r["wind_speed_max"], r["wind_direction_sin_mean"], r["wind_direction_cos_mean"],
        r["max_temp_max"], r["min_temp_min"], r["humidity_min"], r["pdsi_mean"], r["erc_mean"],
    )
    for r in feature_rows
]
con.executemany(
    "INSERT OR REPLACE INTO early_features VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
    insert_rows,
)
con.commit()

train_df = con.execute("""
    SELECT e.year, f.*, o.severity_class
    FROM early_features f
    JOIN events e ON e.event_id = f.event_id
    JOIN event_outcomes o ON o.event_id = f.event_id
""").df()
print(train_df.shape)
train_df.head()

(607, 14)


,year,event_id,cutoff_day,fire_extent_ha,wind_speed_mean,wind_speed_max,wind_direction_sin_mean,wind_direction_cos_mean,max_temp_max,min_temp_min,humidity_min,pdsi_mean,erc_mean,severity_class
0,2018,fire_21458798,2,0.0,0.895985,2.4,-0.354610,-0.385744,294.899994,274.100006,0.00178,-1.642182,52.257618,0
1,2018,fire_21458801,2,0.0,4.423371,9.5,-0.023865,-0.802851,292.399994,259.299988,0.00131,-2.819414,61.060646,0
2,2018,fire_21458806,2,0.0,1.746247,4.0,0.027330,0.047859,295.200012,272.100006,0.00349,-1.862331,26.161974,1
3,2018,fire_21458836,2,112.5,1.051939,2.6,-0.341254,-0.491015,299.399994,271.799988,0.00073,-1.110331,64.648163,0
4,2018,fire_21458848,2,0.0,4.598395,9.1,-0.122574,-0.069806,287.000000,250.300003,0.00114,-2.819055,49.521889,0


## Step 4: Year-based train/test split (not random)

Per the paper's own recommendation: yearly distributions vary a lot (2019
is a known outlier -- fewer, smaller fires), so a random shuffle risks an
unrepresentative split. Hold out one full year as the test set instead --
e.g. train on 2018/2020/2021, test on 2019, or run this across a few
held-out years to sanity check stability before committing to one split.

In [5]:
FEATURE_COLS = [
    "fire_extent_ha", "wind_speed_mean", "wind_speed_max",
    "wind_direction_sin_mean", "wind_direction_cos_mean",
    "max_temp_max", "min_temp_min", "humidity_min", "pdsi_mean", "erc_mean",
]

TEST_YEAR = 2019  # known outlier year in this dataset -- a good stress test

train = train_df[train_df["year"] != TEST_YEAR]
test = train_df[train_df["year"] == TEST_YEAR]

X_train, y_train = train[FEATURE_COLS], train["severity_class"]
X_test, y_test = test[FEATURE_COLS], test["severity_class"]

print(f"Train: {len(train)} events, Test: {len(test)} events")
print("Train class balance:\n", y_train.value_counts().sort_index())
print("Test class balance:\n", y_test.value_counts().sort_index())

Train: 533 events, Test: 74 events
Train class balance:
 severity_class
0    125
1    123
2    138
3    147
Name: count, dtype: int64
Test class balance:
 severity_class
0    30
1    26
2    14
3     4
Name: count, dtype: int64


## Step 5: XGBoost baseline

In [6]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

xgb_clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    objective="multi:softmax", num_class=4, eval_metric="mlogloss",
    random_state=0,
)
xgb_clf.fit(X_train, y_train)

y_pred = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.61      0.47      0.53        30
           1       0.25      0.08      0.12        26
           2       0.19      0.36      0.25        14
           3       0.00      0.00      0.00         4

    accuracy                           0.28        74
   macro avg       0.26      0.23      0.22        74
weighted avg       0.37      0.28      0.30        74

[[14  4  8  4]
 [ 5  2 11  8]
 [ 3  1  5  5]
 [ 1  1  2  0]]


## Step 6: EBM baseline (fully interpretable, glass-box)

Trained on the exact same features and split, so its accuracy and its
native explanations are directly comparable to XGBoost + SHAP.

In [7]:
from interpret.glassbox import ExplainableBoostingClassifier

ebm_clf = ExplainableBoostingClassifier(random_state=0)
ebm_clf.fit(X_train, y_train)

y_pred_ebm = ebm_clf.predict(X_test)
print(classification_report(y_test, y_pred_ebm))

/Users/pmacias/opt/anaconda3/envs/flashpoint/lib/python3.11/site-packages/interpret/glassbox/_ebm/_ebm.py:1249: UserWarning: For multiclass we cannot currently visualize pairs and they will be stripped from the global explanations. Set interactions=0 to generate a fully interpretable glassbox model.
  warn(


              precision    recall  f1-score   support

           0       0.64      0.47      0.54        30
           1       0.20      0.08      0.11        26
           2       0.20      0.36      0.26        14
           3       0.00      0.00      0.00         4

    accuracy                           0.28        74
   macro avg       0.26      0.23      0.23        74
weighted avg       0.37      0.28      0.31        74



In [8]:
from sklearn.metrics import accuracy_score, f1_score

years = sorted(train_df["year"].unique())
results = []

for test_year in years:
    tr = train_df[train_df["year"] != test_year]
    te = train_df[train_df["year"] == test_year]
    Xtr, ytr = tr[FEATURE_COLS], tr["severity_class"]
    Xte, yte = te[FEATURE_COLS], te["severity_class"]

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        objective="multi:softmax", num_class=4, eval_metric="mlogloss", random_state=0,
    )
    clf.fit(Xtr, ytr)
    preds = clf.predict(Xte)

    results.append({
        "test_year": test_year,
        "n_test": len(te),
        "accuracy": accuracy_score(yte, preds),
        "macro_f1": f1_score(yte, preds, average="macro"),
    })

results_df = pd.DataFrame(results)
print(results_df)
print(f"\nMean accuracy: {results_df['accuracy'].mean():.3f}")
print(f"Mean macro F1: {results_df['macro_f1'].mean():.3f}")

   test_year  n_test  accuracy  macro_f1
0       2018     176  0.346591  0.349383
1       2019      74  0.283784  0.223987
2       2020     201  0.293532  0.286710
3       2021     156  0.448718  0.443487

Mean accuracy: 0.343
Mean macro F1: 0.326


In [9]:
import shap

xgb_full = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    objective="multi:softmax", num_class=4, eval_metric="mlogloss", random_state=0,
)
xgb_full.fit(train_df[FEATURE_COLS], train_df["severity_class"])

explainer = shap.TreeExplainer(xgb_full)
shap_values = explainer.shap_values(train_df[FEATURE_COLS])

shap.summary_plot(shap_values, train_df[FEATURE_COLS], plot_type="bar")

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.